# D$_2$O$_2$ controlled isotope test



In [1]:
from pathlib import Path
import hashlib
import json
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.constants import physical_constants, c
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import CubicSpline, PchipInterpolator
from scipy.linalg import eigh
from scipy.optimize import minimize_scalar

HARTREE_TO_CM = physical_constants["hartree-inverse meter relationship"][0] / 100.0
AMU_TO_ME = physical_constants["atomic mass constant"][0] / physical_constants["electron mass"][0]
ANGSTROM_TO_BOHR = 1.0 / physical_constants["Bohr radius"][0] / 1.0e10
CM_TO_HZ = c * 100.0

MASS_AMU = {
    "H": 1.00782503223,
    "D": 2.01410177812,
    "O": 15.99491461957,
}

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIX Two Text", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.0,
    "axes.labelsize": 9.0,
    "axes.titlesize": 10.0,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 8.0,
    "axes.linewidth": 0.8,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLORS = {"H2O2": "#1976B9", "D2O2": "#D55E00"}

In [2]:
PROJECT_ROOT = Path(
    os.environ.get("H2X2_PROJECT_ROOT", Path.cwd() / "H2X2_publication_project")
).resolve()

H2O2_ROOT = PROJECT_ROOT / "data" / "H2O2"
GEOMETRY_ROOT = H2O2_ROOT
GEOMETRY_GLOB = "**/*.xyz"  # Narrow this if H2O2_ROOT contains unrelated XYZ files.
DVR_REFERENCE_PATH = H2O2_ROOT / "processed" / "dvr_wavefunctions.npz"
ALPHA_PATH = H2O2_ROOT / "processed" / "polarizability_surface.npz"

# Set this explicitly if the potential is not stored in DVR_REFERENCE_PATH.
# Accepted formats: NPZ or CSV with a physical torsional angle and V in hartree or cm^-1.
POTENTIAL_PATH = None
POTENTIAL_UNIT_OVERRIDE = None  # None, "hartree", or "cm-1"

PRODUCTION_INPUTS_CONFIRMED = (
    os.environ.get("H2X2_PRODUCTION_INPUTS_CONFIRMED", "1") == "1"
)

DVR_GRIDS = (192, 256, 384, 512)
PRODUCTION_GRID = 384
N_METRIC_POINTS = 4001
TRANS_GUARD_DEG = 5.0
GUARD_SENSITIVITY_DEG = (2.5, 5.0, 7.5, 10.0)
N_STATES_TO_SAVE = 12

# Existing production values are validation targets, not inputs to the isotope calculation.
REFERENCE_H2O2_SPLIT_CM1 = 10.984
REFERENCE_H2O2_A01_AU = 1.565223
REFERENCE_H2O2_CALCULATED_LADDER_CM1 = np.array(
    [0.0, 10.984, 257.3, 371.7, 571.0, 777.3]
)
REFERENCE_H2O2_EXPERIMENTAL_RMS_CM1 = 1.61

# Observed D2O2 torsional origins transcribed from the source identified below.
OBSERVED_D2O2_LADDER_CM1 = np.array(
    [0.0, 1.88, 208.6, 250.9, 387.7, 511.2]
)
D2O2_LADDER_SOURCE = (
    "Hunt and Leacock, J. Chem. Phys. 45, 3141-3147 (1966), Table IV"
)
# Set True only after checking the level assignments directly in Table IV.
D2O2_TABLE_IV_ASSIGNMENTS_VERIFIED = False
SIX_LEVEL_REPRODUCTION_TOLERANCE_CM1 = 2.0

RESULTS_DIR = PROJECT_ROOT / "results" / "D2O2_isotope_test"
FIGURE_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GEOMETRY_ROOT:", GEOMETRY_ROOT)
print("ALPHA_PATH:", ALPHA_PATH)
print("DVR_REFERENCE_PATH:", DVR_REFERENCE_PATH)
print("PRODUCTION_INPUTS_CONFIRMED:", PRODUCTION_INPUTS_CONFIRMED)

PROJECT_ROOT: /home/smohtash/H2X2_final/H2X2_publication_project
GEOMETRY_ROOT: /home/smohtash/H2X2_final/H2X2_publication_project/data/H2O2
ALPHA_PATH: /home/smohtash/H2X2_final/H2X2_publication_project/data/H2O2/processed/polarizability_surface.npz
DVR_REFERENCE_PATH: /home/smohtash/H2X2_final/H2X2_publication_project/data/H2O2/processed/dvr_wavefunctions.npz
PRODUCTION_INPUTS_CONFIRMED: True


In [3]:
def sha256(path, block=1 << 20):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def first_key(archive, names):
    for name in names:
        if name in archive.files:
            return name
    return None


def wrap_pi(values):
    return (np.asarray(values, float) + np.pi) % (2 * np.pi) - np.pi


def read_angle_from_npz(archive):
    key = first_key(archive, (
        "phi_rad", "phi_grid_rad", "phi_of_chi_rad", "physical_phi_rad",
        "phi_dvr_rad", "phi_chi_rad",
    ))
    if key is not None:
        return np.asarray(archive[key], float), key
    key = first_key(archive, ("phi_deg", "phi_grid_deg", "physical_phi_deg"))
    if key is not None:
        return np.radians(np.asarray(archive[key], float)), key
    raise KeyError("No physical torsional-angle array was found")


def read_xyz(path):
    lines = Path(path).read_text().splitlines()
    n = int(lines[0].strip())
    symbols, xyz = [], []
    for line in lines[2:2+n]:
        fields = line.split()
        symbols.append(fields[0])
        xyz.append([float(fields[1]), float(fields[2]), float(fields[3])])
    if len(symbols) != n:
        raise ValueError(f"Incomplete XYZ file: {path}")
    return symbols, np.asarray(xyz, float)


def unit(vector, label="vector"):
    norm = np.linalg.norm(vector)
    if norm < 1e-12:
        raise ValueError(f"Cannot normalize {label}")
    return vector / norm


def identify_hooh(symbols, xyz):
    hidx = [i for i, symbol in enumerate(symbols) if symbol.upper() in ("H", "D")]
    oidx = [i for i, symbol in enumerate(symbols) if symbol.upper() == "O"]
    if len(hidx) != 2 or len(oidx) != 2:
        raise ValueError(f"Expected H2O2 connectivity, found {symbols}")
    o1, o2 = oidx
    direct = np.linalg.norm(xyz[hidx[0]] - xyz[o1]) + np.linalg.norm(xyz[hidx[1]] - xyz[o2])
    crossed = np.linalg.norm(xyz[hidx[1]] - xyz[o1]) + np.linalg.norm(xyz[hidx[0]] - xyz[o2])
    h1, h2 = (hidx[0], hidx[1]) if direct <= crossed else (hidx[1], hidx[0])
    return h1, o1, o2, h2


def dihedral_rad(a, b, c_, d):
    b0 = a - b
    b1 = c_ - b
    b2 = d - c_
    bhat = unit(b1, "central bond")
    v = b0 - np.dot(b0, bhat) * bhat
    w = b2 - np.dot(b2, bhat) * bhat
    return np.arctan2(np.dot(np.cross(bhat, v), w), np.dot(v, w))


def canonical_h1_pinned_geometry(symbols, xyz):
    h1, o1, o2, h2 = identify_hooh(symbols, xyz)
    order = np.array([h1, o1, o2, h2], int)
    xyz = np.asarray(xyz, float)[order]
    origin = 0.5 * (xyz[1] + xyz[2])
    z = unit(xyz[2] - xyz[1], "O-O axis")
    h1_bond = xyz[0] - xyz[1]
    x = unit(h1_bond - np.dot(h1_bond, z) * z, "H1-pinned x axis")
    y = unit(np.cross(z, x), "H1-pinned y axis")
    x = unit(np.cross(y, z), "orthogonalized x axis")
    B = np.column_stack([x, y, z])
    canonical = (xyz - origin) @ B
    phi = dihedral_rad(canonical[0], canonical[1], canonical[2], canonical[3])
    return canonical, float(wrap_pi(phi))


def geometry_files(root):
    files = []
    for path in Path(root).glob(GEOMETRY_GLOB):
        low = str(path).lower()
        if not any(token in low for token in ("processed", "cache", "polarizability_cache")):
            files.append(path)
    return sorted(set(files))


def deduplicate_sorted(x, y, tol=1e-8):
    order = np.argsort(x)
    x = np.asarray(x, float)[order]
    y = np.asarray(y)[order]
    xs, ys = [], []
    start = 0
    for stop in range(1, len(x) + 1):
        if stop == len(x) or abs(x[stop] - x[start]) > tol:
            xs.append(np.mean(x[start:stop]))
            ys.append(np.mean(y[start:stop], axis=0))
            start = stop
    return np.asarray(xs), np.asarray(ys)


def potential_candidates():
    candidates = [DVR_REFERENCE_PATH]
    patterns = ("*potential*.npz", "*potential*.csv", "*torsion*.npz", "*torsion*.csv")
    for pattern in patterns:
        candidates.extend(H2O2_ROOT.rglob(pattern))
    return [p for p in dict.fromkeys(map(Path, candidates)) if p.exists() and "polariz" not in p.name.lower()]


def read_potential(path, unit_override=None):
    path = Path(path)
    if path.suffix.lower() == ".npz":
        with np.load(path, allow_pickle=False) as z:
            phi, phi_key = read_angle_from_npz(z)
            key = first_key(z, (
                "V_ha", "V_hartree", "V_grid_ha", "V_phi_ha", "potential_ha",
                "potential_hartree", "V_chi_ha", "V_cm1", "V_cm-1", "potential_cm1",
            ))
            if key is None:
                raise KeyError(f"No potential array in {path}")
            values = np.asarray(z[key], float)
        inferred = "cm-1" if "cm" in key.lower() else "hartree"
        return phi, values, unit_override or inferred, f"{path}:{phi_key},{key}"
    table = pd.read_csv(path)
    phi_key = next((k for k in ("phi_rad", "phi_deg", "torsional_angle_deg") if k in table), None)
    value_key = next((k for k in ("V_ha", "V_hartree", "V_cm1", "V_cm-1", "energy") if k in table), None)
    if phi_key is None or value_key is None:
        raise KeyError(f"Could not identify angle and potential columns in {path}")
    phi = table[phi_key].to_numpy(float)
    if "deg" in phi_key:
        phi = np.radians(phi)
    if unit_override is None and value_key == "energy":
        raise ValueError("Set POTENTIAL_UNIT_OVERRIDE for a generic 'energy' column")
    inferred = "cm-1" if "cm" in value_key.lower() else "hartree"
    return phi, table[value_key].to_numpy(float), unit_override or inferred, f"{path}:{phi_key},{value_key}"


def resolve_potential_path():
    if POTENTIAL_PATH is not None:
        return Path(POTENTIAL_PATH)
    usable = []
    for path in potential_candidates():
        try:
            read_potential(path, POTENTIAL_UNIT_OVERRIDE)
            usable.append(path)
        except Exception:
            pass
    if len(usable) == 1:
        return usable[0]
    if not usable:
        raise FileNotFoundError(
            "No potential surface was found. Set POTENTIAL_PATH to the production H2O2 potential."
        )
    raise RuntimeError("Multiple potential files are usable; set POTENTIAL_PATH explicitly:\n" + "\n".join(map(str, usable)))

In [4]:
xyz_paths = geometry_files(GEOMETRY_ROOT)
if not xyz_paths:
    raise FileNotFoundError(f"No relaxed H2O2 XYZ geometries found beneath {GEOMETRY_ROOT}")
if not ALPHA_PATH.exists():
    raise FileNotFoundError(f"Missing production polarizability surface: {ALPHA_PATH}")

resolved_potential_path = resolve_potential_path()

inventory = pd.DataFrame({
    "input": ["relaxed geometry tree", "potential", "polarizability", "reference DVR"],
    "path": [str(GEOMETRY_ROOT), str(resolved_potential_path), str(ALPHA_PATH), str(DVR_REFERENCE_PATH)],
    "exists": [GEOMETRY_ROOT.exists(), resolved_potential_path.exists(), ALPHA_PATH.exists(), DVR_REFERENCE_PATH.exists()],
})
display(inventory)
print(f"Discovered {len(xyz_paths)} XYZ geometries")

if not PRODUCTION_INPUTS_CONFIRMED:
    raise RuntimeError(
        "Review the inventory above, then set PRODUCTION_INPUTS_CONFIRMED=True "
        "(or H2X2_PRODUCTION_INPUTS_CONFIRMED=1) and rerun."
    )

,input,path,exists
0,relaxed geometry tree,/home/smohtash/H2X2_final/H2X2_publication_pro...,True
1,potential,/home/smohtash/H2X2_final/H2X2_publication_pro...,True
2,polarizability,/home/smohtash/H2X2_final/H2X2_publication_pro...,True
3,reference DVR,/home/smohtash/H2X2_final/H2X2_publication_pro...,True


Discovered 45 XYZ geometries


In [5]:
def build_periodic_geometry_spline(paths):
    q_values, positive_geometries, source_rows = [], [], []
    for path in paths:
        symbols, xyz = read_xyz(path)
        try:
            geom, phi = canonical_h1_pinned_geometry(symbols, xyz)
        except Exception as exc:
            warnings.warn(f"Skipping {path}: {exc}")
            continue
        # Map every explicit branch to +phi before averaging symmetry equivalents.
        if phi < 0:
            geom = geom.copy()
            geom[:, 1] *= -1.0
        q = abs(phi)
        if np.isclose(q, np.pi, atol=np.radians(0.2)):
            q = np.pi
        if np.isclose(q, 0.0, atol=np.radians(0.2)):
            q = 0.0
        q_values.append(q)
        positive_geometries.append(geom)
        source_rows.append({"file": str(path), "phi_deg": np.degrees(phi), "mapped_q_deg": np.degrees(q)})

    q, coords = deduplicate_sorted(np.asarray(q_values), np.asarray(positive_geometries), tol=np.radians(0.08))
    if len(q) < 12:
        raise RuntimeError(f"Only {len(q)} unique geometries survived; the relaxed path is incomplete")
    if q[0] > np.radians(1.0) or q[-1] < np.radians(179.0):
        raise RuntimeError(
            f"Geometry path must cover cis through trans; found {np.degrees(q[0]):.2f} to {np.degrees(q[-1]):.2f} deg"
        )
    q[0], q[-1] = 0.0, np.pi

    # Complete the negative branch by the physical well reflection y -> -y.
    phi_full = np.r_[-q[:0:-1], q]
    neg = coords[:0:-1].copy()
    neg[:, :, 1] *= -1.0
    coords_full = np.concatenate([neg, coords], axis=0)
    # -pi and +pi are the same planar structure; enforce periodic closure at roundoff level.
    endpoint = 0.5 * (coords_full[0] + coords_full[-1])
    coords_full[0] = endpoint
    coords_full[-1] = endpoint
    spline = CubicSpline(phi_full, coords_full, axis=0, bc_type="periodic")
    return spline, q, coords, pd.DataFrame(source_rows)


def rotation_projected_metric(geometry_spline, phi, terminal_mass):
    phi = np.asarray(phi, float)
    coords = geometry_spline(phi) * ANGSTROM_TO_BOHR
    tangent = geometry_spline(phi, 1) * ANGSTROM_TO_BOHR
    masses = np.array([terminal_mass, MASS_AMU["O"], MASS_AMU["O"], terminal_mass])
    total_mass = masses.sum()

    com = np.einsum("a,nai->ni", masses, coords) / total_mass
    com_velocity = np.einsum("a,nai->ni", masses, tangent) / total_mass
    r = coords - com[:, None, :]
    u = tangent - com_velocity[:, None, :]

    metric = np.empty(len(phi))
    omega_norm = np.empty(len(phi))
    residual = np.empty(len(phi))
    eye = np.eye(3)
    for k in range(len(phi)):
        J = np.zeros((3, 3))
        ell = np.zeros(3)
        for mass, ra, ua in zip(masses, r[k], u[k]):
            J += mass * (np.dot(ra, ra) * eye - np.outer(ra, ra))
            ell += mass * np.cross(ra, ua)
        omega = np.linalg.solve(J, ell)
        uint = u[k] - np.cross(omega[None, :], r[k])
        metric[k] = np.sum(masses[:, None] * uint**2)
        omega_norm[k] = np.linalg.norm(omega)
        residual[k] = np.linalg.norm(J @ omega - ell)
    return metric, omega_norm, residual


geometry_spline, q_geometry, positive_geometry, geometry_inventory = build_periodic_geometry_spline(xyz_paths)
display(geometry_inventory.sort_values("mapped_q_deg").reset_index(drop=True).head())
print("Unique positive-branch geometries:", len(q_geometry))

phi_metric = np.linspace(-np.pi, np.pi, N_METRIC_POINTS, endpoint=False)
metric_data = {}
for label, terminal_mass in (("H2O2", MASS_AMU["H"]), ("D2O2", MASS_AMU["D"])):
    Ieff, omega_norm, residual = rotation_projected_metric(geometry_spline, phi_metric, terminal_mass)
    if not np.all(np.isfinite(Ieff)) or np.min(Ieff) <= 0:
        raise RuntimeError(f"{label}: nonpositive or nonfinite internal metric")
    metric_data[label] = {
        "phi": phi_metric,
        "Ieff": Ieff,
        "omega_norm": omega_norm,
        "projection_residual": residual,
    }

metric_audit = pd.DataFrame([
    {
        "isotopologue": label,
        "min_Ieff_amu_bohr2": data["Ieff"].min(),
        "max_Ieff_amu_bohr2": data["Ieff"].max(),
        "max_projection_residual": data["projection_residual"].max(),
        "finite_positive": bool(np.all(np.isfinite(data["Ieff"])) and np.min(data["Ieff"]) > 0),
    }
    for label, data in metric_data.items()
]).set_index("isotopologue")
display(metric_audit)

,file,phi_deg,mapped_q_deg
0,/home/smohtash/H2X2_final/H2X2_publication_pro...,0.0,0.0
1,/home/smohtash/H2X2_final/H2X2_publication_pro...,0.0,0.0
2,/home/smohtash/H2X2_final/H2X2_publication_pro...,2.5,2.5
3,/home/smohtash/H2X2_final/H2X2_publication_pro...,5.0,5.0
4,/home/smohtash/H2X2_final/H2X2_publication_pro...,10.0,10.0


Unique positive-branch geometries: 44


,min_Ieff_amu_bohr2,max_Ieff_amu_bohr2,max_projection_residual,finite_positive
isotopologue,,,,
H2O2,1.424973,1.549298,1.087792e-15,True
D2O2,2.591009,2.923166,2.175584e-15,True


In [6]:
def even_profile(phi_raw, values_raw, unit="hartree"):
    phi = wrap_pi(phi_raw)
    values = np.asarray(values_raw, float)
    if unit.lower() in ("cm-1", "cm1", "wavenumber"):
        values = values / HARTREE_TO_CM
    elif unit.lower() not in ("hartree", "ha", "au", "a.u."):
        raise ValueError(f"Unsupported potential unit: {unit}")
    q, values = deduplicate_sorted(np.abs(phi), values, tol=1e-8)
    values = values - np.nanmin(values)
    if q[0] > np.radians(1.0) or q[-1] < np.radians(179.0):
        raise RuntimeError("Potential does not span cis through trans")
    q[0], q[-1] = 0.0, np.pi
    interpolator = PchipInterpolator(q, values, extrapolate=False)
    return lambda angle: np.asarray(interpolator(np.abs(wrap_pi(angle))), float)


def load_alpha_yz(path):
    with np.load(path, allow_pickle=False) as z:
        phi, phi_key = read_angle_from_npz(z)
        tensor_key = first_key(z, ("alpha_bf_au", "polarizability_bf_au", "alpha_au"))
        if tensor_key is None:
            raise KeyError("No body-fixed polarizability tensor was found")
        tensor = np.asarray(z[tensor_key], float)
    if tensor.shape != (len(phi), 3, 3):
        raise ValueError(f"Expected alpha tensor shape (n,3,3), found {tensor.shape}")
    tensor_symmetry_error = np.max(abs(tensor - np.swapaxes(tensor, 1, 2)))
    tensor = 0.5 * (tensor + np.swapaxes(tensor, 1, 2))
    return wrap_pi(phi), tensor[:, 1, 2], phi_key, tensor_key, tensor_symmetry_error


def branch_interpolator(q, values):
    q, values = deduplicate_sorted(q, values, tol=1e-8)
    if len(q) < 4:
        raise RuntimeError("Too few points for branch interpolation")
    return PchipInterpolator(q, values, extrapolate=False), q.min(), q.max()


def build_odd_alpha_profile(phi_raw, ayz_raw, guard_deg):
    phi = wrap_pi(phi_raw)
    guard = np.radians(float(guard_deg))
    trusted = np.abs(phi) <= np.pi - guard + 1e-12
    pos = trusted & (phi > 1e-10)
    neg = trusted & (phi < -1e-10)
    if np.count_nonzero(pos) < 4 or np.count_nonzero(neg) < 4:
        raise RuntimeError("Explicit positive and negative alpha_yz branches are required")

    fp, pmin, pmax = branch_interpolator(phi[pos], ayz_raw[pos])
    fn, nmin, nmax = branch_interpolator(-phi[neg], -ayz_raw[neg])
    lo, hi = max(pmin, nmin), min(pmax, nmax)
    audit_q = np.linspace(lo, hi, 1001)
    raw_odd_max_error = float(np.max(abs(fp(audit_q) - fn(audit_q))))

    q_data = np.r_[phi[pos], -phi[neg]]
    a_data = np.r_[ayz_raw[pos], -ayz_raw[neg]]
    q_data, a_data = deduplicate_sorted(q_data, a_data, tol=1e-8)
    q_nodes = np.r_[0.0, q_data, np.pi]
    a_nodes = np.r_[0.0, a_data, 0.0]
    q_nodes, a_nodes = deduplicate_sorted(q_nodes, a_nodes, tol=1e-8)
    positive = PchipInterpolator(q_nodes, a_nodes, extrapolate=False)

    def evaluate(angle):
        p = wrap_pi(angle)
        q = np.abs(p)
        result = positive(q) * np.where(p < 0, -1.0, 1.0)
        result[np.isclose(p, 0.0, atol=1e-13)] = 0.0
        result[np.isclose(q, np.pi, atol=1e-13)] = 0.0
        return np.asarray(result, float)

    return {
        "evaluate": evaluate,
        "guard_deg": float(guard_deg),
        "raw_odd_max_error_au": raw_odd_max_error,
    }


phi_v, V_raw, V_unit, potential_source = read_potential(resolved_potential_path, POTENTIAL_UNIT_OVERRIDE)
V_of_phi = even_profile(phi_v, V_raw, V_unit)

phi_alpha_raw, ayz_raw, alpha_phi_key, alpha_tensor_key, tensor_symmetry_error = load_alpha_yz(ALPHA_PATH)
alpha_profiles = {
    guard: build_odd_alpha_profile(phi_alpha_raw, ayz_raw, guard)
    for guard in GUARD_SENSITIVITY_DEG
}
alpha_profile = alpha_profiles[TRANS_GUARD_DEG]

# The production gauche minimum is fixed by the finalized H2O2 calculation.
PRODUCTION_PHI_EQ_DEG = 112.166769
EXPECTED_ALPHA_EQ_AU = 1.581417
ALPHA_EQ_TOLERANCE_AU = 5.0e-6

# Retain the minimum independently found in this notebook only as a diagnostic.
minimum = minimize_scalar(
    lambda x: float(V_of_phi(np.array([x]))[0]),
    bounds=(np.radians(5.0), np.radians(175.0)),
    method="bounded",
    options={"xatol": 1e-13},
)
phi_eq_notebook = float(minimum.x)

# Use the established production minimum for every reported property.
phi_eq = np.radians(PRODUCTION_PHI_EQ_DEG)
alpha_eq_signed = float(
    alpha_profile["evaluate"](np.array([phi_eq]))[0]
)
alpha_eq = abs(alpha_eq_signed)

phi_eq_difference_deg = (
    np.degrees(phi_eq_notebook) - PRODUCTION_PHI_EQ_DEG
)

# If this fails, the notebook is not using the finalized production
# polarizability file. Inspect ALPHA_PATH and its SHA-256 hash.
if not np.isclose(
    alpha_eq,
    EXPECTED_ALPHA_EQ_AU,
    rtol=0.0,
    atol=ALPHA_EQ_TOLERANCE_AU,
):
    raise RuntimeError(
        "Polarizability provenance mismatch: "
        f"alpha_yz({PRODUCTION_PHI_EQ_DEG:.6f} deg) = "
        f"{alpha_eq:.9f} a.u., expected "
        f"{EXPECTED_ALPHA_EQ_AU:.9f} a.u.\n"
        f"ALPHA_PATH = {ALPHA_PATH}\n"
        f"SHA256 = {sha256(ALPHA_PATH)}"
    )

In [7]:
def periodic_scalar_interpolator(phi, values):
    order = np.argsort(phi)
    phi = np.asarray(phi)[order]
    values = np.asarray(values)[order]
    # Append the duplicated endpoint required by a periodic cubic spline.
    x = np.r_[phi, np.pi]
    y = np.r_[values, values[0]]
    return CubicSpline(x, y, bc_type="periodic")


def arc_mapping(phi_metric, Ieff):
    I_spline = periodic_scalar_interpolator(phi_metric, Ieff)
    q = np.linspace(0.0, np.pi, N_METRIC_POINTS // 2 + 1)
    Iq = I_spline(q)
    if np.min(Iq) <= 0:
        raise RuntimeError("Interpolated metric became nonpositive")
    s = cumulative_trapezoid(np.sqrt(Iq), q, initial=0.0)
    s_pi = float(s[-1])
    phi_of_s = PchipInterpolator(s, q, extrapolate=False)
    Ichi = (s_pi / np.pi) ** 2

    def phi_of_chi(chi):
        chi = np.asarray(chi, float)
        q_out = phi_of_s(np.abs(chi) * s_pi / np.pi)
        return np.where(chi < 0, -q_out, q_out)

    return phi_of_chi, float(Ichi)


def reflection_basis(phi):
    phi = wrap_pi(phi)
    distance = np.abs(wrap_pi(phi[None, :] + phi[:, None]))
    mapping = np.argmin(distance, axis=1)
    error = float(np.max(distance[np.arange(len(phi)), mapping]))
    if not np.array_equal(mapping[mapping], np.arange(len(phi))):
        raise RuntimeError("Reflection map is not an involution")
    used = np.zeros(len(phi), bool)
    even_columns, odd_columns = [], []
    for i, j in enumerate(mapping):
        if used[i]:
            continue
        used[i] = used[j] = True
        if i == j:
            column = np.zeros(len(phi))
            column[i] = 1.0
            even_columns.append(column)
        else:
            ep = np.zeros(len(phi)); ep[i] = ep[j] = 1 / np.sqrt(2)
            em = np.zeros(len(phi)); em[i] = 1 / np.sqrt(2); em[j] = -1 / np.sqrt(2)
            even_columns.append(ep)
            odd_columns.append(em)
    return np.column_stack(even_columns), np.column_stack(odd_columns), mapping, error


def fourier_dvr(label, n_grid, alpha_evaluator=alpha_profile["evaluate"]):
    data = metric_data[label]
    phi_of_chi, Ichi = arc_mapping(data["phi"], data["Ieff"])
    chi = np.linspace(-np.pi, np.pi, n_grid, endpoint=False)
    phi = phi_of_chi(chi)
    V = V_of_phi(phi)
    V = V - V.min()

    modes = np.fft.fftfreq(n_grid, d=1.0 / n_grid)
    F = np.exp(1j * np.outer(chi, modes)) / np.sqrt(n_grid)
    kinetic_eigenvalues = modes**2 / (2.0 * Ichi * AMU_TO_ME)
    T = (F * kinetic_eigenvalues[None, :]) @ F.conj().T
    H = np.real_if_close(T, tol=1000).real + np.diag(V)

    Qe, Qo, reflection, reflection_error = reflection_basis(chi)
    He = Qe.T @ H @ Qe
    Ho = Qo.T @ H @ Qo
    Ee, Ce = eigh(He, subset_by_index=(0, min(N_STATES_TO_SAVE - 1, He.shape[0] - 1)))
    Eo, Co = eigh(Ho, subset_by_index=(0, min(N_STATES_TO_SAVE - 1, Ho.shape[0] - 1)))
    Pe = Qe @ Ce
    Po = Qo @ Co
    even = Pe[:, 0]
    odd = Po[:, 0]
    alpha = alpha_evaluator(phi)
    A = np.vdot(even, alpha * odd)
    if abs(A) < 1e-14:
        raise RuntimeError(f"{label}: Raman matrix element is numerically zero")
    odd = odd * np.exp(-1j * np.angle(A))
    Po = Po * np.exp(-1j * np.angle(A))
    A = np.vdot(even, alpha * odd)
    contribution = np.real(np.conj(even) * alpha * odd)

    split_ha = float(Eo[0] - Ee[0])
    split_cm1 = split_ha * HARTREE_TO_CM
    first_transfer_s = 1.0 / (2.0 * CM_TO_HZ * abs(split_cm1))
    trans = np.abs(phi) > np.pi - np.radians(TRANS_GUARD_DEG)
    trans_fraction = np.sum(np.abs(contribution[trans])) / np.sum(np.abs(contribution))
    localized_left = (even + odd) / np.sqrt(2)
    localized_right = (even - odd) / np.sqrt(2)
    alpha_left = float(np.vdot(localized_left, alpha * localized_left).real)
    alpha_right = float(np.vdot(localized_right, alpha * localized_right).real)

    reflected_H = H[np.ix_(reflection, reflection)]
    commutator_relative = np.linalg.norm(H - reflected_H) / np.linalg.norm(H)
    return {
        "label": label, "n_grid": n_grid, "chi": chi, "phi": phi, "V_ha": V,
        "Ichi_amu_bohr2": Ichi, "Ee_ha": Ee, "Eo_ha": Eo, "Pe": Pe, "Po": Po,
        "even": even, "odd": odd, "localized_left": localized_left,
        "localized_right": localized_right, "alpha_grid_au": alpha,
        "contribution": contribution, "split_ha": split_ha, "split_cm1": split_cm1,
        "frequency_Hz": abs(split_cm1) * CM_TO_HZ,
        "first_transfer_s": first_transfer_s, "A01_complex_au": A,
        "A01_au": float(abs(A)), "A01_over_alpha_eq": float(abs(A) / alpha_eq),
        "alpha_left_au": alpha_left, "alpha_right_au": alpha_right,
        "localized_identity_error_au": abs(float(A.real) - alpha_left),
        "even_parity": float(np.vdot(even, even[reflection]).real),
        "odd_parity": float(np.vdot(odd, odd[reflection]).real),
        "state_overlap_abs": float(abs(np.vdot(even, odd))),
        "reflection_grid_error_rad": reflection_error,
        "hamiltonian_reflection_error": commutator_relative,
        "quadrature_reconstruction_error": float(abs(np.sum(contribution) - A.real)),
        "matrix_element_imaginary_residual": float(abs(A.imag)),
        "trans_abs_contribution_fraction": float(trans_fraction),
    }


convergence_rows = []
all_solutions = {}
for n_grid in DVR_GRIDS:
    for label in ("H2O2", "D2O2"):
        solution = fourier_dvr(label, n_grid)
        all_solutions[(label, n_grid)] = solution
        convergence_rows.append({
            "isotopologue": label,
            "n_grid": n_grid,
            "Ichi_amu_bohr2": solution["Ichi_amu_bohr2"],
            "split_cm-1": solution["split_cm1"],
            "A01_au": solution["A01_au"],
            "A01_over_alpha_eq": solution["A01_over_alpha_eq"],
        })

convergence = pd.DataFrame(convergence_rows).set_index(["isotopologue", "n_grid"])
display(convergence)
solutions = {label: all_solutions[(label, PRODUCTION_GRID)] for label in ("H2O2", "D2O2")}

,,Ichi_amu_bohr2,split_cm-1,A01_au,A01_over_alpha_eq
isotopologue,n_grid,,,,
H2O2,192,1.502491,10.985056,1.565216,0.989756
D2O2,192,2.784122,1.628214,1.581091,0.999794
H2O2,256,1.502491,10.985132,1.565220,0.989758
D2O2,256,2.784122,1.628161,1.581092,0.999795
H2O2,384,1.502491,10.985041,1.565221,0.989759
D2O2,384,2.784122,1.628182,1.581093,0.999795
H2O2,512,1.502491,10.985038,1.565221,0.989759
D2O2,512,2.784122,1.628182,1.581093,0.999795


In [8]:
summary_rows = []
for label, solution in solutions.items():
    summary_rows.append({
        "isotopologue": label,
        "Ichi_amu_bohr2": solution["Ichi_amu_bohr2"],
        "Delta01_cm-1": solution["split_cm1"],
        "frequency_Hz": solution["frequency_Hz"],
        "first_L_to_R_time_s": solution["first_transfer_s"],
        "alpha_eq_au": alpha_eq,
        "A01_yz_au": solution["A01_au"],
        "A01_over_alpha_eq": solution["A01_over_alpha_eq"],
        "trans_abs_contribution_fraction": solution["trans_abs_contribution_fraction"],
    })
summary = pd.DataFrame(summary_rows).set_index("isotopologue")

comparison = pd.Series({
    "H_to_D_splitting_suppression_factor": abs(summary.loc["H2O2", "Delta01_cm-1"] / summary.loc["D2O2", "Delta01_cm-1"]),
    "H_to_D_transfer_time_increase_factor": summary.loc["D2O2", "first_L_to_R_time_s"] / summary.loc["H2O2", "first_L_to_R_time_s"],
    "D_over_H_A01": summary.loc["D2O2", "A01_yz_au"] / summary.loc["H2O2", "A01_yz_au"],
    "A01_percent_change_H_to_D": 100 * (summary.loc["D2O2", "A01_yz_au"] / summary.loc["H2O2", "A01_yz_au"] - 1),
    "D2O2_deviation_from_1p88_percent": 100 * (
        summary.loc["D2O2", "Delta01_cm-1"] / OBSERVED_D2O2_LADDER_CM1[1] - 1
    ),
}, name="controlled isotope comparison")

display(summary)
display(comparison)

,Ichi_amu_bohr2,Delta01_cm-1,frequency_Hz,first_L_to_R_time_s,alpha_eq_au,A01_yz_au,A01_over_alpha_eq,trans_abs_contribution_fraction
isotopologue,,,,,,,,
H2O2,1.502491,10.985041,3.293232e+11,1.518265e-12,1.581417,1.565221,0.989759,0.000990
D2O2,2.784122,1.628182,4.881168e+10,1.024345e-11,1.581417,1.581093,0.999795,0.000276


H_to_D_splitting_suppression_factor      6.746813
H_to_D_transfer_time_increase_factor     6.746813
D_over_H_A01                             1.010140
A01_percent_change_H_to_D                1.014013
D2O2_deviation_from_1p88_percent       -13.394562
Name: controlled isotope comparison, dtype: float64

In [9]:
def merged_parity_ladder(solution, n_levels=6):
    energies_ha = np.r_[solution["Ee_ha"], solution["Eo_ha"]]
    parity = np.array(["+" for _ in solution["Ee_ha"]] + ["-" for _ in solution["Eo_ha"]])
    within_parity_index = np.r_[
        np.arange(len(solution["Ee_ha"])), np.arange(len(solution["Eo_ha"]))
    ]
    order = np.argsort(energies_ha)
    energies_ha = energies_ha[order]
    parity = parity[order]
    within_parity_index = within_parity_index[order]
    relative_cm1 = (energies_ha - energies_ha[0]) * HARTREE_TO_CM
    return pd.DataFrame({
        "level_index": np.arange(n_levels),
        "parity": parity[:n_levels],
        "index_within_parity_sector": within_parity_index[:n_levels],
        "calculated_cm-1": relative_cm1[:n_levels],
    })


ladder_frames = []
ladder_results = {}
for label in ("H2O2", "D2O2"):
    ladder = merged_parity_ladder(solutions[label], n_levels=6)
    if label == "H2O2":
        reference = REFERENCE_H2O2_CALCULATED_LADDER_CM1
        reference_kind = "previous manuscript calculation"
    else:
        reference = OBSERVED_D2O2_LADDER_CM1
        reference_kind = "experiment (Hunt and Leacock, Table IV)"
    ladder["reference_cm-1"] = reference
    ladder["residual_calc_minus_ref_cm-1"] = ladder["calculated_cm-1"] - reference
    ladder["isotopologue"] = label
    ladder["reference_kind"] = reference_kind
    ladder_results[label] = ladder
    ladder_frames.append(ladder)

six_level_comparison = pd.concat(ladder_frames, ignore_index=True).set_index(
    ["isotopologue", "level_index"]
)
h2_ladder_rms = float(np.sqrt(np.mean(
    ladder_results["H2O2"]["residual_calc_minus_ref_cm-1"]**2
)))
d2_ladder_rms = float(np.sqrt(np.mean(
    ladder_results["D2O2"]["residual_calc_minus_ref_cm-1"]**2
)))
d2_excited_only_rms = float(np.sqrt(np.mean(
    ladder_results["D2O2"].loc[1:, "residual_calc_minus_ref_cm-1"]**2
)))

ladder_audit = pd.Series({
    "H2O2_RMS_vs_previous_calculated_ladder_cm-1": h2_ladder_rms,
    "H2O2_previous_experimental_RMS_reported_in_manuscript_cm-1": REFERENCE_H2O2_EXPERIMENTAL_RMS_CM1,
    "D2O2_RMS_vs_observed_six_levels_cm-1": d2_ladder_rms,
    "D2O2_RMS_vs_observed_excited_levels_only_cm-1": d2_excited_only_rms,
    "calculated_isotope_splitting_ratio_H_over_D": abs(
        ladder_results["H2O2"].loc[1, "calculated_cm-1"] /
        ladder_results["D2O2"].loc[1, "calculated_cm-1"]
    ),
    "observed_isotope_splitting_ratio_H_over_D": 11.44 / OBSERVED_D2O2_LADDER_CM1[1],
    "H2O2_six_level_reproduction_PASS": bool(
        h2_ladder_rms < SIX_LEVEL_REPRODUCTION_TOLERANCE_CM1
    ),
    "D2O2_Table_IV_assignments_verified": bool(D2O2_TABLE_IV_ASSIGNMENTS_VERIFIED),
}, name="six-level ladder audit")

display(six_level_comparison)
display(ladder_audit)

parity  index_within_parity_sector  calculated_cm-1  \
isotopologue level_index                                                       
H2O2         0                +                           0         0.000000   
             1                -                           0        10.985041   
             2                +                           1       257.313401   
             3                -                           1       371.663900   
             4                +                           2       571.017630   
             5                -                           2       777.272138   
D2O2         0                +                           0         0.000000   
             1                -                           0         1.628182   
             2                +                           1       215.012018   
             3                -                           1       253.377080   
             4                +                           2       392.723778   
             5                -                           2       513.568341   

                          reference_cm-1  residual_calc_minus_ref_cm-1  \
isotopologue level_index                                                 
H2O2         0                     0.000                      0.000000   
             1                    10.984                      0.001041   
             2                   257.300                      0.013401   
             3                   371.700                     -0.036100   
             4                   571.000                      0.017630   
             5                   777.300                     -0.027862   
D2O2         0                     0.000                      0.000000   
             1                     1.880                     -0.251818   
             2                   208.600                      6.412018   
             3                   250.900                      2.477080   
             4                   387.700                      5.023778   
             5                   511.200                      2.368341   

                                                   reference_kind  
isotopologue level_index                                           
H2O2         0                    previous manuscript calculation  
             1                    previous manuscript calculation  
             2                    previous manuscript calculation  
             3                    previous manuscript calculation  
             4                    previous manuscript calculation  
             5                    previous manuscript calculation  
D2O2         0            experiment (Hunt and Leacock, Table IV)  
             1            experiment (Hunt and Leacock, Table IV)  
             2            experiment (Hunt and Leacock, Table IV)  
             3            experiment (Hunt and Leacock, Table IV)  
             4            experiment (Hunt and Leacock, Table IV)  
             5            experiment (Hunt and Leacock, Table IV)

H2O2_RMS_vs_previous_calculated_ladder_cm-1                     0.0207
H2O2_previous_experimental_RMS_reported_in_manuscript_cm-1        1.61
D2O2_RMS_vs_observed_six_levels_cm-1                          3.609264
D2O2_RMS_vs_observed_excited_levels_only_cm-1                  3.95375
calculated_isotope_splitting_ratio_H_over_D                   6.746813
observed_isotope_splitting_ratio_H_over_D                     6.085106
H2O2_six_level_reproduction_PASS                                  True
D2O2_Table_IV_assignments_verified                               False
Name: six-level ladder audit, dtype: object

In [10]:
guard_rows = []
for guard, profile in alpha_profiles.items():
    for label in ("H2O2", "D2O2"):
        tested = fourier_dvr(label, PRODUCTION_GRID, alpha_evaluator=profile["evaluate"])
        guard_rows.append({
            "isotopologue": label,
            "trans_guard_deg": guard,
            "A01_au": tested["A01_au"],
            "A01_over_alpha_eq": tested["A01_au"] / alpha_eq,
            "trans_abs_contribution_fraction": tested["trans_abs_contribution_fraction"],
            "raw_alpha_odd_error_au": profile["raw_odd_max_error_au"],
        })
guard_sensitivity = pd.DataFrame(guard_rows).set_index(["isotopologue", "trans_guard_deg"])
display(guard_sensitivity)

audit_rows = []
for label, solution in solutions.items():
    subset = convergence.loc[label]
    split_spread = (subset["split_cm-1"].max() - subset["split_cm-1"].min()) / abs(subset["split_cm-1"].iloc[-1])
    A_spread = (subset["A01_au"].max() - subset["A01_au"].min()) / subset["A01_au"].iloc[-1]
    guards = guard_sensitivity.loc[label]
    guard_spread = (guards["A01_au"].max() - guards["A01_au"].min()) / guards["A01_au"].iloc[1]
    row = {
        "isotopologue": label,
        "metric_finite_positive": bool(metric_audit.loc[label, "finite_positive"]),
        "even_parity_error": abs(solution["even_parity"] - 1.0),
        "odd_parity_error": abs(solution["odd_parity"] + 1.0),
        "state_overlap_abs": solution["state_overlap_abs"],
        "reflection_grid_error_rad": solution["reflection_grid_error_rad"],
        "hamiltonian_reflection_error": solution["hamiltonian_reflection_error"],
        "quadrature_reconstruction_error": solution["quadrature_reconstruction_error"],
        "matrix_element_imaginary_residual": solution["matrix_element_imaginary_residual"],
        "localized_identity_error_au": solution["localized_identity_error_au"],
        "relative_grid_spread_split": split_spread,
        "relative_grid_spread_A01": A_spread,
        "relative_guard_spread_A01": guard_spread,
        "trans_abs_contribution_fraction": solution["trans_abs_contribution_fraction"],
    }
    row["numerical_PASS"] = bool(
        row["metric_finite_positive"]
        and row["even_parity_error"] < 1e-12
        and row["odd_parity_error"] < 1e-12
        and row["state_overlap_abs"] < 1e-12
        and row["hamiltonian_reflection_error"] < 1e-10
        and row["quadrature_reconstruction_error"] < 1e-12
        and row["localized_identity_error_au"] < 1e-10
        and row["relative_grid_spread_split"] < 2e-3
        and row["relative_grid_spread_A01"] < 2e-3
        and row["relative_guard_spread_A01"] < 5e-3
    )
    audit_rows.append(row)

audit = pd.DataFrame(audit_rows).set_index("isotopologue")
h_reference_error = abs(summary.loc["H2O2", "Delta01_cm-1"] / REFERENCE_H2O2_SPLIT_CM1 - 1)
h_A_error = abs(summary.loc["H2O2", "A01_yz_au"] / REFERENCE_H2O2_A01_AU - 1)
reference_audit = pd.Series({
    "H2O2_split_relative_error_vs_production": h_reference_error,
    "H2O2_A01_relative_error_vs_production": h_A_error,
    "H2O2_six_level_RMS_vs_production_cm-1": h2_ladder_rms,
    "H2O2_six_level_reproduction_PASS": bool(
        h2_ladder_rms < SIX_LEVEL_REPRODUCTION_TOLERANCE_CM1
    ),
    "H2O2_reference_reproduction_PASS": bool(h_reference_error < 5e-3 and h_A_error < 5e-3),
    "all_numerical_checks_PASS": bool(audit["numerical_PASS"].all()),
}, name="submission gate")

display(audit)
display(reference_audit)

,,A01_au,A01_over_alpha_eq,trans_abs_contribution_fraction,raw_alpha_odd_error_au
isotopologue,trans_guard_deg,,,,
H2O2,2.5,1.565498,0.989934,0.001167,5.551115e-16
D2O2,2.5,1.581172,0.999845,0.000326,5.551115e-16
H2O2,5.0,1.565221,0.989759,0.000990,5.551115e-16
D2O2,5.0,1.581093,0.999795,0.000276,5.551115e-16
H2O2,7.5,1.564166,0.989092,0.000617,6.661338e-16
D2O2,7.5,1.580783,0.999599,0.000172,6.661338e-16
H2O2,10.0,1.564166,0.989092,0.000617,6.661338e-16
D2O2,10.0,1.580783,0.999599,0.000172,6.661338e-16


,metric_finite_positive,even_parity_error,odd_parity_error,state_overlap_abs,reflection_grid_error_rad,hamiltonian_reflection_error,quadrature_reconstruction_error,matrix_element_imaginary_residual,localized_identity_error_au,relative_grid_spread_split,relative_grid_spread_A01,relative_guard_spread_A01,trans_abs_contribution_fraction,numerical_PASS
isotopologue,,,,,,,,,,,,,,
H2O2,True,2.220446e-16,5.551115e-16,5.095036e-19,8.881784e-16,6.840131e-14,2.220446e-16,1.916843e-16,2.220446e-16,0.000009,0.000003,0.000851,0.000990,True
D2O2,True,0.000000e+00,2.220446e-16,5.068632e-19,8.881784e-16,6.837172e-14,0.000000e+00,1.936280e-16,4.440892e-16,0.000033,0.000001,0.000246,0.000276,True


H2O2_split_relative_error_vs_production    0.000095
H2O2_A01_relative_error_vs_production      0.000001
H2O2_six_level_RMS_vs_production_cm-1        0.0207
H2O2_six_level_reproduction_PASS               True
H2O2_reference_reproduction_PASS               True
all_numerical_checks_PASS                      True
Name: submission gate, dtype: object

In [11]:
surface_audit = pd.Series({
    "potential_source": potential_source,
    "alpha_source": f"{ALPHA_PATH}:{alpha_phi_key},{alpha_tensor_key}",
    "alpha_file_sha256": sha256(ALPHA_PATH),
    "phi_eq_notebook_deg": np.degrees(phi_eq_notebook),
    "phi_eq_production_deg": PRODUCTION_PHI_EQ_DEG,
    "phi_eq_difference_deg": phi_eq_difference_deg,
    "alpha_eq_signed_au": alpha_eq_signed,
    "alpha_eq_abs_au": alpha_eq,
    "tensor_symmetry_error_au": tensor_symmetry_error,
    "raw_alpha_yz_odd_error_au":
        alpha_profile["raw_odd_max_error_au"],
}, name="fixed electronic surfaces")

display(surface_audit)

potential_source             /home/smohtash/H2X2_final/H2X2_publication_pro...
alpha_source                 /home/smohtash/H2X2_final/H2X2_publication_pro...
alpha_file_sha256            0cdd3cc846cc80abe2888667f3dec270374b142747b044...
phi_eq_notebook_deg                                                 112.488403
phi_eq_production_deg                                               112.166769
phi_eq_difference_deg                                                 0.321634
alpha_eq_signed_au                                                    1.581417
alpha_eq_abs_au                                                       1.581417
tensor_symmetry_error_au                                                   0.0
raw_alpha_yz_odd_error_au                                                  0.0
Name: fixed electronic surfaces, dtype: object

## 9. Publication-quality isotope-test figures

The figure tells the controlled-experiment story in three steps: deuteration changes the internal kinetic metric; the heavier isotope becomes more concentrated in the same wells on the same potential; and the tunneling splitting changes far more strongly than the Raman matrix element.

In [14]:
def format_time(seconds):
    if seconds < 1e-9:
        return f"{seconds*1e12:.4g} ps"
    if seconds < 1e-6:
        return f"{seconds*1e9:.4g} ns"
    if seconds < 1e-3:
        return f"{seconds*1e6:.4g} \\mu s"
    return f"{seconds:.4g} s"


summary.to_csv(RESULTS_DIR / "D2O2_isotope_summary.csv")
convergence.to_csv(RESULTS_DIR / "D2O2_DVR_convergence.csv")
guard_sensitivity.to_csv(RESULTS_DIR / "D2O2_trans_guard_sensitivity.csv")
audit.to_csv(RESULTS_DIR / "D2O2_numerical_audit.csv")
metric_audit.to_csv(RESULTS_DIR / "D2O2_metric_audit.csv")
geometry_inventory.to_csv(RESULTS_DIR / "D2O2_geometry_inventory.csv", index=False)
six_level_comparison.to_csv(RESULTS_DIR / "H2O2_D2O2_six_level_comparison.csv")
ladder_audit.to_csv(RESULTS_DIR / "H2O2_D2O2_six_level_audit.csv")

for label, solution in solutions.items():
    np.savez_compressed(
        RESULTS_DIR / f"{label}_isotope_test_DVR.npz",
        chi_rad=solution["chi"], phi_rad=solution["phi"], V_ha=solution["V_ha"],
        Ichi_amu_bohr2=solution["Ichi_amu_bohr2"],
        Ee_ha=solution["Ee_ha"], Eo_ha=solution["Eo_ha"],
        Pe=solution["Pe"], Po=solution["Po"], alpha_yz_au=solution["alpha_grid_au"],
    )

tex_lines = [
    r"\begin{table}[t]",
    r"\caption{Controlled isotope test using a common H$_2$O$_2$ Born--Oppenheimer potential and polarizability surface.}",
    r"\label{tab:d2o2_isotope}",
    r"\begin{ruledtabular}",
    r"\begin{tabular}{lccccc}",
    r"Isotopologue & $I_\chi$ (amu bohr$^2$) & $\Delta_{01}$ (cm$^{-1}$) & $t_{L\to R}^{(1)}$ & $|A_{01}^{yz}|$ (a.u.) & $|A_{01}^{yz}|/|\alpha_{yz}(\phi_{\rm eq})|$ \\",
    r"\hline",
]
for label in ("H2O2", "D2O2"):
    row = summary.loc[label]
    molecule = r"H$_2$O$_2$" if label == "H2O2" else r"D$_2$O$_2$"
    tex_lines.append(
        f"{molecule} & {row.Ichi_amu_bohr2:.4f} & {row['Delta01_cm-1']:.7g} & "
        f"{format_time(row.first_L_to_R_time_s)} & {row.A01_yz_au:.4f} & "
        f"{row.A01_over_alpha_eq:.4f} \\\\"
    )
tex_lines += [r"\end{tabular}", r"\end{ruledtabular}", r"\end{table}"]
(RESULTS_DIR / "D2O2_isotope_table.tex").write_text("\n".join(tex_lines) + "\n")

ladder_tex = [
    r"\begin{table}[t]",
    r"\caption{Six-level D$_2$O$_2$ torsional benchmark on the unchanged H$_2$O$_2$ Born--Oppenheimer surface.}",
    r"\label{tab:d2o2_ladder}",
    r"\begin{ruledtabular}",
    r"\begin{tabular}{ccrr}",
    r"Level & Parity & Observed (cm$^{-1}$) & Calculated (cm$^{-1}$) \\",
    r"\hline",
]
for _, row in ladder_results["D2O2"].iterrows():
    ladder_tex.append(
        f"{int(row.level_index)} & ${row.parity}$ & {row['reference_cm-1']:.2f} & "
        f"{row['calculated_cm-1']:.3f} \\\\"
    )
ladder_tex += [
    r"\hline",
    rf"\multicolumn{{3}}{{r}}{{Six-level rms deviation}} & {d2_ladder_rms:.3f} \\",
    r"\end{tabular}", r"\end{ruledtabular}", r"\end{table}",
]
(RESULTS_DIR / "D2O2_six_level_benchmark_table.tex").write_text(
    "\n".join(ladder_tex) + "\n"
)

provenance = {
    "project_root": str(PROJECT_ROOT),
    "potential": {"path": str(resolved_potential_path), "sha256": sha256(resolved_potential_path)},
    "polarizability": {"path": str(ALPHA_PATH), "sha256": sha256(ALPHA_PATH)},
    "geometry_files": [{"path": str(path), "sha256": sha256(path)} for path in xyz_paths],
    "masses_amu": MASS_AMU,
    "production_grid": PRODUCTION_GRID,
    "all_numerical_checks_PASS": bool(audit["numerical_PASS"].all()),
    "H2O2_reference_reproduction_PASS": bool(reference_audit["H2O2_reference_reproduction_PASS"]),
    "H2O2_six_level_reproduction_PASS": bool(reference_audit["H2O2_six_level_reproduction_PASS"]),
    "D2O2_ladder_source": D2O2_LADDER_SOURCE,
    "D2O2_Table_IV_assignments_verified": bool(D2O2_TABLE_IV_ASSIGNMENTS_VERIFIED),
}
(RESULTS_DIR / "D2O2_input_provenance.json").write_text(json.dumps(provenance, indent=2))

FINAL_PASS = bool(
    audit["numerical_PASS"].all()
    and reference_audit["H2O2_reference_reproduction_PASS"]
    and reference_audit["H2O2_six_level_reproduction_PASS"]
    and PRODUCTION_INPUTS_CONFIRMED
)
CITATION_READY = bool(FINAL_PASS and D2O2_TABLE_IV_ASSIGNMENTS_VERIFIED)
print("FINAL SUBMISSION GATE:", "PASS" if FINAL_PASS else "FAIL")
if not FINAL_PASS:
    print("Do not quote the D2O2 numbers until the failed audit item is resolved.")
else:
    print("Production inputs confirmed; H2O2 ground and six-level results reproduced.")
print("D2O2 TABLE-IV CITATION GATE:", "PASS" if CITATION_READY else "VERIFY SOURCE ASSIGNMENTS")
print("Results:", RESULTS_DIR)
print("Figure:", figure_base.with_suffix(".pdf"))
print("Six-level figure:", ladder_figure_base.with_suffix(".pdf"))

FINAL SUBMISSION GATE: PASS
Production inputs confirmed; H2O2 ground and six-level results reproduced.
D2O2 TABLE-IV CITATION GATE: VERIFY SOURCE ASSIGNMENTS
Results: /home/smohtash/H2X2_final/H2X2_publication_project/results/D2O2_isotope_test
Figure: /home/smohtash/H2X2_final/H2X2_publication_project/figures/Fig_D2O2_controlled_isotope_test.pdf
Six-level figure: /home/smohtash/H2X2_final/H2X2_publication_project/figures/Fig_D2O2_six_level_benchmark.pdf
